# Évaluation Finale — Résultats pour le Mémoire

Ce notebook génère toutes les figures et tableaux pour le Chapitre 5.

In [ ]:
import sys
import os
import time
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.models.base_models import create_logistic_regression, create_decision_tree
from src.models.ensemble_models import (
    create_random_forest, create_xgboost, create_lightgbm, create_catboost
)
from src.models.stacking_model import FraudStackingEnsemble
from src.models.deep_models import MLPFraudClassifier, AutoencoderDetector
from src.evaluation.metrics import (
    compute_all_metrics, compute_classification_report, compute_inference_time
)
from src.evaluation.visualizer import (
    plot_roc_curves, plot_pr_curves, plot_confusion_matrix,
    plot_confusion_matrices_grid, plot_model_comparison_bar
)
from src.evaluation.comparator import ModelComparator
from src.evaluation.cost_analysis import CostBenefitAnalyzer
from config import MODELS_DIR, FIGURES_DIR, DATA_PROCESSED_DIR

print('Imports OK')

In [ ]:
# Charger les données prétraitées
X_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_train.csv'))
X_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'X_test.csv'))
y_train = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_train.csv')).values.ravel()
y_test = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, 'y_test.csv')).values.ravel()

# Essayer de charger les modèles sauvegardés, sinon ré-entraîner
print(f'Données: Train={X_train.shape}, Test={X_test.shape}')
print(f'Taux de fraude test: {y_test.mean()*100:.3f}%')

# Définir les 8 modèles
model_configs = {
    'Logistic Regression': lambda: create_logistic_regression(),
    'Decision Tree': lambda: create_decision_tree(),
    'Random Forest': lambda: create_random_forest(),
    'XGBoost': lambda: create_xgboost(),
    'LightGBM': lambda: create_lightgbm(),
    'CatBoost': lambda: create_catboost(),
    'MLP': lambda: MLPFraudClassifier(input_dim=X_train.shape[1]),
    'Stacking': lambda: FraudStackingEnsemble(),
}

print(f'\n{len(model_configs)} modèles à évaluer')

## Tableau Comparatif Global

In [ ]:
# Entraîner et évaluer les 8 modèles
comparator = ModelComparator()
all_probas = {}  # Pour les courbes ROC/PR
all_preds = {}   # Pour les matrices de confusion

for name, create_fn in model_configs.items():
    print(f'\n--- {name} ---')
    model = create_fn()
    
    t0 = time.time()
    
    # Gestion spéciale pour l'Autoencoder et le MLP
    if name == 'MLP':
        class_weight = {0: 1, 1: (y_train == 0).sum() / max((y_train == 1).sum(), 1)}
        model.fit(X_train.values, y_train, epochs=50, batch_size=256,
                  class_weight=class_weight)
        t_train = time.time() - t0
        y_proba_i = model.predict(X_test.values)
        y_pred_i = model.predict_classes(X_test.values)
    else:
        # Stacking et modèles sklearn
        if name == 'Stacking':
            model.fit(X_train, y_train)
            t_train = time.time() - t0
            y_proba_i = model.predict_proba(X_test)[:, 1]
            y_pred_i = model.predict(X_test)
        else:
            model.fit(X_train, y_train)
            t_train = time.time() - t0
            y_proba_i = model.predict_proba(X_test)[:, 1]
            y_pred_i = model.predict(X_test)
    
    metrics_i = compute_all_metrics(y_test, y_pred_i, y_proba_i)
    metrics_i['train_time_s'] = round(t_train, 2)
    comparator.add_result(name, metrics_i)
    
    all_probas[name] = y_proba_i
    all_preds[name] = y_pred_i
    
    print(f'  AUC-ROC: {metrics_i["auc_roc"]:.4f} | F1: {metrics_i["f1_score"]:.4f} | '
          f'Precision: {metrics_i["precision"]:.4f} | Recall: {metrics_i["recall"]:.4f} | '
          f'Temps: {t_train:.1f}s')

print('\n=== Tous les modèles évalués ===')

In [ ]:
# Tableau comparatif global
comparison_df = comparator.get_comparison_table()

display_cols = ['auc_roc', 'auprc', 'f1_score', 'precision', 'recall',
                'specificity', 'accuracy', 'train_time_s', 'inference_time_ms']
available_cols = [c for c in display_cols if c in comparison_df.columns]

styled = comparison_df[available_cols].style\
    .highlight_max(
        subset=[c for c in ['auc_roc', 'auprc', 'f1_score', 'precision', 'recall'] if c in available_cols],
        color='#a8e6cf'
    )\
    .highlight_min(
        subset=[c for c in ['train_time_s'] if c in available_cols],
        color='#dcedc1'
    )\
    .format('{:.4f}', subset=[c for c in available_cols if c not in ['train_time_s', 'inference_time_ms']])\
    .set_caption('Tableau 5.1 — Comparaison globale des 8 modèles')

styled

# Sauvegarder en CSV
results_dir = os.path.join(PROJECT_ROOT, 'reports', 'results')
os.makedirs(results_dir, exist_ok=True)
comparison_df[available_cols].to_csv(os.path.join(results_dir, 'final_comparison.csv'))
print(f'Résultats sauvegardés: {os.path.join(results_dir, "final_comparison.csv")}')

In [ ]:
# Export du tableau en LaTeX pour le mémoire
latex_table = comparator.export_latex_table(
    metrics=['auc_roc', 'auprc', 'f1_score', 'precision', 'recall', 'train_time_s']
)

print('=== Tableau LaTeX (pour le mémoire) ===')
print(latex_table)

# Sauvegarder le LaTeX
latex_path = os.path.join(results_dir, 'comparison_table.tex')
with open(latex_path, 'w') as f:
    f.write(latex_table)
print(f'\nLaTeX sauvegardé: {latex_path}')

## Courbes ROC

In [ ]:
# Courbes ROC pour les 8 modèles
roc_dict = {name: (y_test, proba) for name, proba in all_probas.items()}

roc_path = os.path.join(FIGURES_DIR, 'models', 'roc_curves_all.png')
os.makedirs(os.path.dirname(roc_path), exist_ok=True)
fig_roc = plot_roc_curves(roc_dict, save_path=roc_path)
print(f'Courbes ROC sauvegardées: {roc_path}')

## Courbes Precision-Recall

In [ ]:
# Courbes Precision-Recall pour les 8 modèles
pr_dict = {name: (y_test, proba) for name, proba in all_probas.items()}

pr_path = os.path.join(FIGURES_DIR, 'models', 'pr_curves_all.png')
fig_pr = plot_pr_curves(pr_dict, save_path=pr_path)
print(f'Courbes PR sauvegardées: {pr_path}')

## Matrices de Confusion

In [ ]:
# Grille de matrices de confusion (2x4)
cm_dict = {name: (y_test, pred) for name, pred in all_preds.items()}

cm_grid_path = os.path.join(FIGURES_DIR, 'models', 'confusion_matrices_grid.png')
fig_grid = plot_confusion_matrices_grid(cm_dict, save_path=cm_grid_path)
print(f'Grille de matrices de confusion sauvegardée: {cm_grid_path}')

## Comparaison Grouped Bar Chart

In [ ]:
# Graphique à barres groupées — Modèles x Métriques
bar_path = os.path.join(FIGURES_DIR, 'comparison', 'model_comparison_bar.png')
os.makedirs(os.path.dirname(bar_path), exist_ok=True)

fig_bar = plot_model_comparison_bar(
    comparison_df,
    metrics=['auc_roc', 'f1_score', 'precision', 'recall', 'auprc'],
    save_path=bar_path
)
print(f'Graphique comparatif sauvegardé: {bar_path}')

## Analyse Coût-Bénéfice

In [ ]:
# Analyse coût-bénéfice pour le modèle de stacking
from src.evaluation.cost_analysis import CostBenefitAnalyzer

cba = CostBenefitAnalyzer()

# Analyse de base
y_pred_stacking = all_preds['Stacking']
y_proba_stacking = all_probas['Stacking']

cost_result = cba.analyze(y_test, y_pred_stacking, y_proba_stacking)

print('=== Analyse Coût-Bénéfice — Stacking Ensemble ===')
print(f'\nFraudes détectées (TP):      {cost_result["tp"]:,}')
print(f'Fraudes manquées (FN):       {cost_result["fn"]:,}')
print(f'Fausses alertes (FP):        {cost_result["fp"]:,}')
print(f'\nValeur des fraudes détectées: {cost_result["fraud_caught_value"]:,.2f} EUR')
print(f'Valeur des fraudes manquées:  {cost_result["fraud_missed_value"]:,.2f} EUR')
print(f'Coût des fausses alertes:     {cost_result["fp_cost_total"]:,.2f} EUR')
print(f'Coût d\'investigation:         {cost_result["investigation_cost_total"]:,.2f} EUR')
print(f'\n>>> Bénéfice net:            {cost_result["net_benefit"]:,.2f} EUR')
print(f'>>> Taux de détection:       {cost_result["detection_rate"]:.2f}%')

# Courbe de coût vs seuil
cost_curve_path = os.path.join(FIGURES_DIR, 'comparison', 'cost_curve_stacking.png')
fig_cost = cba.plot_cost_curve(y_test, y_proba_stacking, save_path=cost_curve_path)
print(f'\nCourbe de coût sauvegardée: {cost_curve_path}')

# Seuil optimal
optimal = cba.optimal_threshold(y_test, y_proba_stacking)
print(f'Seuil optimal: {optimal["optimal_threshold"]:.2f}')
print(f'Coût total minimal: {optimal["min_total_cost"]:,.2f} EUR')

## Résumé des Chiffres Clés

Tous les chiffres ci-dessous alimentent le **Chapitre 5** du mémoire.

| Indicateur | Valeur |
|---|---|
| Meilleur modèle (AUC-ROC) | Stacking Ensemble |
| AUC-ROC Stacking | *cf. tableau ci-dessus* |
| AUPRC Stacking | *cf. tableau ci-dessus* |
| F1-Score Stacking | *cf. tableau ci-dessus* |
| Gain Stacking vs meilleur individuel (AUC) | *cf. comparaison* |
| Feature la plus importante (SHAP) | V14 |
| Bénéfice net estimé | *cf. analyse coût-bénéfice* |
| Seuil optimal de classification | *cf. analyse coût-bénéfice* |
| Nombre de modèles évalués | 8 |
| Nombre de features | *cf. données* |

Les figures sont sauvegardées dans `reports/figures/` pour inclusion directe dans le document LaTeX.